In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
from scipy.stats import ttest_ind, chi2_contingency
import warnings
from pandas import MultiIndex #, Int64Index # changed
warnings.filterwarnings('ignore')
import os

# 1. Problem Formulation
Design a ML model that predicts diabetes

## Understand the dataset
Each record / row is a data of a patient.

- Pregnancies: Number of times pregnant
- Glucose: Plasma glucose concentration a 2 hours in an oral glucose tolerance test
- BloodPressure: Diastolic blood pressure (mm Hg)
- SkinThickness: Triceps skin fold thickness (mm)
- Insulin: 2-Hour serum insulin (mu U/ml)
- BMI: Body mass index (weight in kg/(height in m)^2)
- DiabetesPedigreeFunction: Diabetes pedigree function.
  - Its calculated score that estimates a person's likelihood of developing diabetes based on their family history of the disease, essentially acting as a measure of how strong their genetic predisposition to diabetes is; a higher value indicates a greater family history of diabetes and thus a higher potential risk for the individual.
  - This probability realistically ranges of 0.08 to 2.42.
- Age: Age (years)
- Outcome: Class variable (0 or 1)

# 2. Data collection


In [6]:
df = pd.read_csv("Diabetes-dataset_1.csv")
print(df.sample(10))

     Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
125            1       88             30             42       99  55.0   
17             7      107             74              0        0  29.6   
99             1      122             90             51      220  49.7   
524            3      125             58              0        0  31.6   
422            0      102             64             46       78  40.6   
391            5      166             76              0        0  45.7   
181            0      119             64             18       92  34.9   
381            0      105             68             22        0  20.0   
147            2      106             64             35      119  30.5   
497            2       81             72             15       76  30.1   

     DiabetesPedigreeFunction  Age  Outcome  
125                     0.496   26        1  
17                      0.254   31        1  
99                      0.325   31        1  
5

# 3. Data labelling + Fix Data Types 

In [10]:
# I want to see the descriptive stats of all numerical columns
# Set the option to display all columns
pd.set_option('display.max_columns', None)
print(df.describe())
pd.reset_option('display.max_columns')


       Pregnancies     Glucose  BloodPressure  SkinThickness     Insulin  \
count   768.000000  768.000000     768.000000     768.000000  768.000000   
mean      3.845052  120.894531      69.105469      20.536458   79.799479   
std       3.369578   31.972618      19.355807      15.952218  115.244002   
min       0.000000    0.000000       0.000000       0.000000    0.000000   
25%       1.000000   99.000000      62.000000       0.000000    0.000000   
50%       3.000000  117.000000      72.000000      23.000000   30.500000   
75%       6.000000  140.250000      80.000000      32.000000  127.250000   
max      17.000000  199.000000     122.000000      99.000000  846.000000   

              BMI  DiabetesPedigreeFunction         Age     Outcome  
count  768.000000                768.000000  768.000000  768.000000  
mean    31.992578                  0.471876   33.240885    0.348958  
std      7.884160                  0.331329   11.760232    0.476951  
min      0.000000                  

In [9]:
print(df.info(memory_usage='deep'))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB
None


In [8]:
# Lets print memory usage before memory optimization

print(df.memory_usage(deep=True))
print(f"Before optimization: {df.memory_usage(deep=True).sum() / 1024:.2f} KB")

Index                        128
Pregnancies                 6144
Glucose                     6144
BloodPressure               6144
SkinThickness               6144
Insulin                     6144
BMI                         6144
DiabetesPedigreeFunction    6144
Age                         6144
Outcome                     6144
dtype: int64
Before optimization: 54.12 KB


In [ ]:
# Lets drop columns that we do not need: "EmployeeCount", "EmployeeNumber", etc and other columns if needed
# df.drop(columns=["EmployeeCount", "EmployeeNumber", "StandardHours"], inplace=True)

# print(df.memory_usage(deep=True))
# print(f"Memory now: {df.memory_usage(deep=True).sum() / 1024:.2f} KB")

In [11]:
# Downcast integer int64 columns to int8, int16, int32, int64 based on the data range present
int_cols = df.select_dtypes(include=['int64']).columns
df[int_cols] = df[int_cols].apply(pd.to_numeric, downcast='integer')

print(df.memory_usage(deep=True))
print(f"Memory now: {df.memory_usage(deep=True).sum() / 1024:.2f} KB")

Index                        128
Pregnancies                  768
Glucose                     1536
BloodPressure                768
SkinThickness                768
Insulin                     1536
BMI                         6144
DiabetesPedigreeFunction    6144
Age                          768
Outcome                      768
dtype: int64
Memory now: 18.88 KB


In [1]:
# How much saving in  memory ? 65%
(54.12 - 18.88) / 54.12

0.6511456023651145

In [12]:
# Lets verify that downcast worked
print(df.dtypes)

Pregnancies                    int8
Glucose                       int16
BloodPressure                  int8
SkinThickness                  int8
Insulin                       int16
BMI                         float64
DiabetesPedigreeFunction    float64
Age                            int8
Outcome                        int8
dtype: object


In [ ]:
# NOT NEEDED HERE: Convert object to Categorical Text Columns

# obj_cols = df.select_dtypes(include=['object']).columns

# for col in obj_cols:
#     num_unique = df[col].nunique()
#     num_total = len(df[col])
#     if num_unique / num_total < 0.5:  # heuristic: if < 50% unique
#         print(f"Converting {col} to datatype category")
#         df[col] = df[col].astype('category')


# print(df.memory_usage(deep=True))
# print(f"Memory now: {df.memory_usage(deep=True).sum() / 1024:.2f} KB")

In [13]:
# Step2: I save this in pickle file so I can preserve the data structure, data types, etc
# I save the dataset in a pickle file and this way when I load it
# back again  for bivariate analysis, it would preserve the data types

df.to_pickle('data_Diabetes_clean.pkl')